In [3]:
import numpy as np

import pyscf
from pyscf.dft.numint import _dot_ao_dm, _contract_rho

from cc2cc import extend
from cc2cc.utils import gen_basis, Grid, rotate

# acetylene_cc-pVDZ_0-1_1_-0.4000_1_18_18
# acetylene_cc-pVDZ_0-1_1_-0.4000_3_27_114

BASIS = "cc-pVDZ"
LEVEL, PERIOD = 1, 2
INDEX_ = (3, 27, 114)
molecular, name = extend("ethylene", "0-1", 1, -0.5, BASIS)
rotate(molecular)

mol = pyscf.M(atom=molecular, basis=gen_basis(molecular, BASIS, True), spin=0)
print(f"Generate data for {name}")

mf = pyscf.scf.RHF(mol)
mf.kernel()
mycc = pyscf.cc.CCSD(mf)
mycc.kernel()
dm1_cc = mycc.make_rdm1(ao_repr=True)
e_cc = mycc.e_tot

mdft = pyscf.scf.RKS(mol)
mdft.xc = "b3lyp"
mdft.kernel()

grids = Grid(mol, level=LEVEL, period=PERIOD)
shls_slice = (0, mol.nbas)
ao_loc = mol.ao_loc_nr()

LEVEL: 1
PERIOD: 2
MAIN_PATH: /home/chenzihao/workspace/cc2cc
DATA_PATH: /home/chenzihao/workspace/cc2cc/data/grids_dft
DATA_CC_PATH: /home/chenzihao/workspace/cc2cc/data/grids_dft
DATA_SAVE_PATH: /home/chenzihao/workspace/cc2cc/data/grids_dft/saved_data
DATA_TEST_PATH: /home/chenzihao/workspace/cc2cc/data/test
STRUCTURE: cnn3d
TEST: False
CUBE_USE: 5
Generate ethylene_-0.5000
Extend 0-1 1 -0.5000
original mol [['C', -0.6672, 0, 0], ['C', 0.6672, 0, 0], ['H', -1.2213, -0.929, 0.0708], ['H', -1.2212, 0.929, -0.0708], ['H', 1.2213, 0.929, -0.0708], ['H', 1.2213, -0.929, 0.0708]]
extend mol [['C', -0.6672, 0, 0], ['C', 0.16720000000000002, 0.0, 0.0], ['H', -1.2213, -0.929, 0.0708], ['H', -1.2212, 0.929, -0.0708], ['H', 1.2213, 0.929, -0.0708], ['H', 1.2213, -0.929, 0.0708]]
Generate data for ethylene_cc-pVDZ_0-1_1_-0.5000
converged SCF energy = -77.1163204056415


<class 'pyscf.cc.ccsd.CCSD'> does not have attributes  converged


E(CCSD) = -77.41661436696734  E_corr = -0.300293961325852


/home/chenzihao/anaconda3/envs/pyscf/lib/python3.12/site-packages/pyscf/dft/libxc.py:507: UserWarning: Since PySCF-2.3, B3LYP (and B3P86) are changed to the VWN-RPA variant, corresponding to the original definition by Stephens et al. (issue 1480) and the same as the B3LYP functional in Gaussian. To restore the VWN5 definition, you can put the setting "B3LYP_WITH_VWN5 = True" in pyscf_conf.py
  warnings.warn('Since PySCF-2.3, B3LYP (and B3P86) are changed to the VWN-RPA variant, '


converged SCF energy = -77.6997452127422
n_rad: 40, n_ang: 194


In [4]:
import opt_einsum as oe

dm2_cc = mycc.make_rdm2(ao_repr=True)
expr_rinv_dm2_r = oe.contract_expression(
    "ijkl,i,j,kl->",
    0.5 * dm2_cc
    - 0.5 * oe.contract("pq,rs->pqrs", dm1_cc, dm1_cc)
    + 0.05 * oe.contract("pr,qs->pqrs", dm1_cc, dm1_cc),
    (mol.nao,),
    (mol.nao,),
    (mol.nao, mol.nao),
    constants=[0],
    optimize="optimal",
)

x_mat = grids.vector_to_matrix(grids.coords[:, 0])
y_mat = grids.vector_to_matrix(grids.coords[:, 1])
z_mat = grids.vector_to_matrix(grids.coords[:, 2])

for i, coord in enumerate([[x_mat[INDEX_], y_mat[INDEX_], z_mat[INDEX_]]]):
    ao_i = pyscf.dft.numint.eval_ao(mol, [coord], deriv=2)
    rho_cc = pyscf.dft.numint.eval_rho(mol, ao_i, dm1_cc, xctype="GGA")
    ao_0_i = ao_i[0, 0]
    exc_over_dm_cc_i = -pyscf.dft.libxc.eval_xc("b3lyp", rho_cc)[0]
    exc_over_dm_b3lyp_i = exc_over_dm_cc_i.copy()

    with mol.with_rinv_origin(coord):
        rinv = mol.intor("int1e_rinv")
        print(
            627.509
            * (
                exc_over_dm_cc_i
                + expr_rinv_dm2_r(
                    ao_0_i,
                    ao_0_i,
                    rinv,
                    backend="torch",
                )
                / rho_cc[0]
            )
        )

    rho_cc_1 = np.zeros((3))
    rho_cc_2 = np.zeros((3, 3))

    # # Hessian matrix
    # assert (
    #     np.linalg.norm(dm1_cc.conj().T - dm1_cc) < 1e-10
    # ), "Density matrix is not symmetric."
    # c0 = _dot_ao_dm(mol, ao_i[0][0], dm1_cc, None, shls_slice, ao_loc)
    # rho_cc_1[0] = _contract_rho(ao_i[0][1], c0)
    # rho_cc_1[1] = _contract_rho(ao_i[0][2], c0)
    # rho_cc_1[2] = _contract_rho(ao_i[0][3], c0)
    # rho_cc_2[0, 0] = _contract_rho(ao_i[0][4], c0)
    # rho_cc_2[0, 1] = _contract_rho(ao_i[0][5], c0)
    # rho_cc_2[0, 2] = _contract_rho(ao_i[0][6], c0)
    # rho_cc_2[1, 1] = _contract_rho(ao_i[0][7], c0)
    # rho_cc_2[1, 2] = _contract_rho(ao_i[0][8], c0)
    # rho_cc_2[2, 2] = _contract_rho(ao_i[0][9], c0)
    # rho_cc_2[1, 0] = rho_cc_2[0, 1]
    # rho_cc_2[2, 0] = rho_cc_2[0, 2]
    # rho_cc_2[2, 1] = rho_cc_2[1, 2]

[-11.43040693]


In [34]:
# methane_cc-pVDZ_0-1_1_-0.5000_3_36_29
data1 = np.load("../data/grids_dft/data_methane_cc-pVDZ_0-1_1_-0.5000_1_2.npz")
INDEX_1 = (3, 36, 29)

# methane_cc-pVDZ_0-1_1_-0.3000_3_36_29
data2 = np.load("../data/grids_dft/data_methane_cc-pVDZ_0-1_1_-0.3000_1_2.npz")
INDEX_2 = (3, 36, 29)

print(
    data1["exc_over_dm_cc_grids"][grids.index_2d[INDEX_1]]
    # * data1["rho_inv_4_norm"][0][grids.index_2d[INDEX_1]]
    # * data1["weights"][grids.index_2d[INDEX_1]]
    * 627.509
)
print(
    data2["exc_over_dm_cc_grids"][grids.index_2d[INDEX_2]]
    # * data1["rho_inv_4_norm"][0][grids.index_2d[INDEX_1]]
    # * data1["weights"][grids.index_2d[INDEX_1]]
    * 627.509
)

input1 = data1["rho_cube"][grids.index_2d[INDEX_1]]
input2 = data2["rho_cube"][grids.index_2d[INDEX_2]]
input1[1, :, :, :] = input1[1, :, :, :] ** (1 / 2)
input2[1, :, :, :] = input2[1, :, :, :] ** (1 / 2)

print(np.linalg.norm(input1 - input2))

6.271649829347757
6.216020455159222
2.3819808394068862e-15


In [ ]:
# data1.files
# 642.7595767022364 - 614.8195523389757

27.940024363260704

6450

In [35]:
INDEX_1

(0, 10, 161)